In [1]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.5/819.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 3.5 MB/s eta 0:00:00


In [2]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051


In [3]:
import os
from pyngrok import ngrok

In [4]:
ngrok.kill()

In [5]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://senior-favorably-fasting.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://senior-favorably-fasting.ngrok-free.dev


True

In [6]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# === 初始化 Google Gemini ===
client = genai.Client(api_key=gemini_api_key)
#這裡讓AI有記憶
chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        response_modalities=["TEXT"],
    )
)

In [7]:
def stateful_query(payload):
    response = chat.send_message(message=payload)
    return response.text

In [8]:
result = stateful_query("簡介明新科技大學")
print(result)

明新科技大學 (Ming Hsin University of Science and Technology, 簡稱明新科大) 是一所位於台灣新竹縣新豐鄉的私立科技大學。學校以「實務應用」為導向，致力於培育產業所需之專業技術人才，特別在工程、管理與服務事業領域擁有深厚的教學與研究基礎。

以下是明新科技大學的簡要介紹：

1.  **創校歷史與發展：**
    *   創立於1966年，前身為「明新工業專科學校」。
    *   經過多年的發展與轉型，於2002年改制為「明新科技大學」。
    *   學校在台灣技職教育體系中扮演重要角色，持續為國家社會培養中高階技術人才。

2.  **地理位置優勢：**
    *   明新科大地理位置優越，緊鄰新竹科學園區、湖口工業區等台灣重要的科技與產業聚落。
    *   這使得學校在推動產學合作、提供學生實習與就業機會方面，具有得天獨厚的優勢。

3.  **學院與學術特色：**
    *   學校目前設有：
        *   **工學院：** 涵蓋精密機械、電子、電機、資訊工程、土木工程與環境資源管理等領域，是學校歷史最悠久、最核心的學院。
        *   **管理學院：** 設有企業管理、資訊管理、行銷與流通管理、財務金融、餐飲管理等系所，培養具備現代管理知識與實務能力的專業人才。
        *   **服務事業與設計學院：** 涵蓋休閒事業管理、旅館事業管理、幼兒保育、多媒體與遊戲發展、時尚造型設計等領域，順應產業變遷，提供多元服務與創意設計人才。
    *   **課程設計：** 強調理論與實務並重，注重學生的動手實作能力，設有完善的實習工廠、實驗室與專業教室。

4.  **教育目標與特色：**
    *   **產學合作：** 與多家知名企業建立密切的產學合作關係，共同研發專案，提供學生職場實習機會，提高畢業生就業競爭力。
    *   **就業導向：** 課程規劃與產業需求緊密連結，致力於培養「畢業即就業，上班即上手」的實用型人才。
    *   **證照取得：** 鼓勵學生考取專業證照，提升專業技能與職場適應力。
    *   **國際化發展：** 積極推動國際交流與合作，提供學生赴海外研習或交換的機會，拓展國際視野。

總體而言，明新科技大學是一所深耕台灣技職教育

In [9]:
result2 = stateful_query("校長是誰？")
print(result2)

截至我知識庫的最新資訊（通常是2023年或更早，因為校長任期可能有變動），**明新科技大學的現任校長是 劉國偉 博士**。

劉國偉校長於2023年8月1日就任，是明新科大第八任校長。他之前曾擔任學校的副校長、總務長、電機工程系主任、研發長等多項重要職務，對學校的發展與運作有著豐富的經驗與深刻的了解。


In [10]:
from flask import Flask, request, abort

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


@app.route("/", methods=['POST'])
def callback():
    # get X-Line-Signature header value
    signature = request.headers['X-Line-Signature']

    # get request body as text
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    # handle webhook body
    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature. Please check your channel access token/channel secret.")
        abort(400)

    return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    text = event.message.text
    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)
        if text.startswith('AI '):
            prompt = text[3:]
            reply_text = stateful_query(prompt)
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=reply_text)]
                )
            )

        else:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=event.message.text),
                        TextMessage(text=event.message.text)]
                )
            )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit


BODY:  {"destination":"Ub7dc1b532060a415e48eb3b397a1484d","events":[{"type":"message","message":{"type":"text","id":"617037752369676289","quoteToken":"DzYUny5fEgdXg0rKk0HxUQVb1yx8iEwkjESg24e7TgC-TZyaSjNL2-zh38xZnDbZZCGET2JEBltBxQPjkOtIyYdp80XayhlOFaG9I0XSEKGQrzMjBq_nuvid8ocbA8sKqiAz4If8TMa3D-fIwqsIAA","markAsReadToken":"tI5Kkj9ERVbEDKy7n7Bcn4xDstO2XsWfA2B0JS_USKVOb2sXg1pMl5KIhzVaEkDZb5BjWNSlMJXk_LSIX9R8GxHUVPVl4WJvezcIoJHuA3spDVIXht12Vfbu9kHFVM4CPcww5xaGWiDwFasHCXUN1BqTnMKi4tSgLEr-4zGTHyPMr6A0V6EsCyED3eIU2PXiZ6zlZk8-zz9SVTtrQkDMkA","text":"AI 介紹明新科技大學，20字以內"},"webhookEventId":"01KTAE91DHJ38P9QX6PT04SC05","deliveryContext":{"isRedelivery":false},"timestamp":1780614464438,"source":{"type":"user","userId":"Ufee3e53621d2838f4ec84bfc9fe80681"},"replyToken":"ff4d6d42e78f47dab3084e5935d51efa","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [04/Jun/2026 23:07:46] "POST / HTTP/1.1" 200 -


BODY:  {"destination":"Ub7dc1b532060a415e48eb3b397a1484d","events":[{"type":"message","message":{"type":"text","id":"617037793524187137","quoteToken":"FGbESe4t1jn3Zw4fio7_oLtNyDnPsPmRQ2sSeScYqq4nR43RaGXVJ34lEleLB2u3DH3MKHpVS49p7VD9S1bY4PH_pIobVZnbDAh1zEIPOJoZV6yWefNvxiCH2sjw8V0yu3bTGFtNRIrlOZWy5FHcIw","markAsReadToken":"0V73ykKqXObB_e5t91IIERxItbC2WFwuCevW8CT4UqTo6x0BYcg_ND5BWz8ukrmt1FSLQbEhiE4fF9KWV5v7tkBk620SNUlUTALZsX3RskfRDR-qKmfQEHwYg4iOBUKbxIevZvuFlKg5iywFpiNKviFSF7myLHq9LEG-zSB1nc3pa4x6E5uzeaQUsD9867Sug-pduKm5EbH5nPaqOGx8ng","text":"AI 現任校長是誰"},"webhookEventId":"01KTAE9SC5YQ3R4891YAKGE7F8","deliveryContext":{"isRedelivery":false},"timestamp":1780614488969,"source":{"type":"user","userId":"Ufee3e53621d2838f4ec84bfc9fe80681"},"replyToken":"9d021150bda84279b348ae477bdb65cd","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [04/Jun/2026 23:08:11] "POST / HTTP/1.1" 200 -
